# Dynamic Pricing for Perishable Inventory

A seller holds **N** units of a perishable good and has **T** periods to sell them.
At the start of each period the seller sets a price from a discrete grid.
Demand is Poisson with a rate that decreases exponentially in price.
Any stock remaining at the end of period T is worthless.
We find the revenue-maximising policy via **backward induction**.

In [ ]:
import numpy as np
from scipy.stats import poisson
import matplotlib.pyplot as plt

## Model

**State** $(t, n)$: period $t \in \{0,\ldots,T-1\}$, remaining inventory $n \in \{0,\ldots,N\}$.

**Demand** at price $p$: $D(p)\sim\mathrm{Poisson}(\lambda(p))$ where $\lambda(p)=a\,e^{-bp}$.

**Bellman equation**
$$V(t,n)=\max_{p\in\mathcal{P}}\;\mathbb{E}\!\left[p\cdot\min(D(p),n)+V\!\left(t+1,\,n-\min(D(p),n)\right)\right]$$

**Boundary conditions**: $V(T,n)=0$ for all $n$ (perishability); $V(t,0)=0$ for all $t$ (no stock).

In [ ]:
T      = 10                        # number of selling periods
N      = 8                         # initial (maximum) inventory
prices = np.arange(1.0, 9.0)       # discrete price grid: 1, 2, ..., 8

def demand_rate(p, a=6.0, b=0.5):
    """Expected Poisson demand as a function of price."""
    return a * np.exp(-b * p)

p_grid = np.linspace(1, 8, 200)
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(p_grid, demand_rate(p_grid))
ax.set_xlabel('Price')
ax.set_ylabel('Expected demand $\\lambda(p)$')
ax.set_title('Demand curve')
plt.tight_layout()
plt.show()

## Backward Induction

Working from $t=T-1$ back to $t=0$, for each state $(t,n)$ we compute the
expected payoff of every candidate price and store the maximiser.

The expectation decomposes over demand realisations $k$:
$$Q(t,n,p)=\sum_{k=0}^{n-1}\Pr(D=k)\bigl(pk+V(t+1,n-k)\bigr)
           +\Pr(D\ge n)\cdot pn$$
where the last term uses $V(t+1,0)=0$.

In [ ]:
V  = np.zeros((T + 1, N + 1))         # V[t, n]
pi = np.zeros((T,     N + 1), dtype=int)  # index into prices

for t in range(T - 1, -1, -1):
    for n in range(1, N + 1):
        best = -np.inf
        for j, p in enumerate(prices):
            lam = demand_rate(p)
            q = sum(
                poisson.pmf(k, lam) * (p * k + V[t + 1, n - k])
                for k in range(n)
            )
            q += poisson.sf(n - 1, lam) * (p * n)   # stock-out: sell all n
            if q > best:
                best, pi[t, n] = q, j
        V[t, n] = best

print('Value function V(0, n) for n = 0 .. N:')
print(np.round(V[0], 2))

## Optimal Value Function

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for n in range(1, N + 1):
    ax.plot(range(T), V[:T, n], marker='o', markersize=3, label=f'n={n}')
ax.set_xlabel('Period t')
ax.set_ylabel('V(t, n)')
ax.set_title('Expected revenue-to-go by period and inventory level')
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

## Optimal Pricing Policy

Two structural properties should be visible in the heatmap:
- **Scarcity premium** — fewer units remaining at the same period → higher price.
- **Deadline discount** — as the horizon approaches with excess stock, the optimal price falls to clear inventory.

In [ ]:
opt_p = prices[pi[:, 1:]]   # shape (T, N): optimal price at each (t, n)

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(
    opt_p.T, origin='lower', aspect='auto',
    extent=[-0.5, T - 0.5, 0.5, N + 0.5],
    cmap='RdYlGn_r', vmin=prices[0], vmax=prices[-1]
)
ax.set_xlabel('Period t')
ax.set_ylabel('Remaining inventory n')
ax.set_title('Optimal price $\\pi(t, n)$')
ax.set_xticks(range(T))
ax.set_yticks(range(1, N + 1))
plt.colorbar(im, ax=ax, label='Price')
plt.tight_layout()
plt.show()

## Monte Carlo Validation

We simulate the optimal policy over many independent episodes.
The sample mean should converge to $V(0, N)$.

In [ ]:
rng = np.random.default_rng(seed=42)
M   = 5_000

revenues = np.zeros(M)
for m in range(M):
    n, total = N, 0.0
    for t in range(T):
        if n == 0:
            break
        p      = prices[pi[t, n]]
        sold   = min(rng.poisson(demand_rate(p)), n)
        total += p * sold
        n     -= sold
    revenues[m] = total

print(f'DP bound  V(0,{N}) = {V[0, N]:.4f}')
print(f'Sim mean          = {revenues.mean():.4f}  (std={revenues.std():.4f})')

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(revenues, bins=40, edgecolor='white', alpha=0.85)
ax.axvline(V[0, N],         color='crimson', linestyle='--', linewidth=1.8,
           label=f'DP value = {V[0, N]:.2f}')
ax.axvline(revenues.mean(), color='navy',    linestyle='--', linewidth=1.8,
           label=f'Sim mean = {revenues.mean():.2f}')
ax.set_xlabel('Total revenue')
ax.set_ylabel('Count')
ax.set_title(f'Revenue distribution  ({M:,} episodes, N={N}, T={T})')
ax.legend()
plt.tight_layout()
plt.show()